In [ ]:
import numpy as np
import pandas as pd
from skclust.metrics import cv_score, eta_squared_score, entropy_score, cramers_v_score

np.random.seed(42)

# ============================================================================
# Test set 1: Perfect clusters (should give best scores across the board)
# Two clusters with completely distinct feature profiles
# ============================================================================
X_perfect_continuous = pd.DataFrame(
    np.vstack([
        np.random.normal(loc=5, scale=0.1, size=(50, 5)),   # Cluster A: tight around 5
        np.random.normal(loc=15, scale=0.1, size=(50, 5)),  # Cluster B: tight around 15
    ]),
    index=[f"s{i}" for i in range(100)],
)

X_perfect_binary = pd.DataFrame(
    np.vstack([
        np.zeros((50, 5), dtype=int),
        np.ones((50, 5), dtype=int),
    ]),
    index=[f"s{i}" for i in range(100)],
)

labels_perfect = pd.Series(
    ["A"] * 50 + ["B"] * 50,
    index=[f"s{i}" for i in range(100)],
)

# ============================================================================
# Test set 2: Random clusters (should give worst scores)
# Labels are shuffled — no relationship between features and clusters
# ============================================================================
labels_random = labels_perfect.sample(frac=1, random_state=0)
labels_random.index = labels_perfect.index

X_random_binary = pd.DataFrame(
    np.random.randint(0, 2, size=(100, 5)),
    index=[f"s{i}" for i in range(100)],
)

# ============================================================================
# Test set 3: Moderate clusters (should give intermediate scores)
# Some signal but with overlap
# ============================================================================
X_moderate_continuous = pd.DataFrame(
    np.vstack([
        np.random.normal(loc=5, scale=2, size=(50, 5)),
        np.random.normal(loc=8, scale=2, size=(50, 5)),
    ]),
    index=[f"s{i}" for i in range(100)],
)

X_moderate_binary = pd.DataFrame(
    np.vstack([
        np.random.binomial(1, 0.2, size=(50, 5)),
        np.random.binomial(1, 0.8, size=(50, 5)),
    ]),
    index=[f"s{i}" for i in range(100)],
)

# ============================================================================
# Run all metrics
# ============================================================================
print("=" * 60)
print("CONTINUOUS METRICS")
print("=" * 60)

for name, X, labels in [
    ("Perfect clusters", X_perfect_continuous, labels_perfect),
    ("Moderate clusters", X_moderate_continuous, labels_perfect),
    ("Random labels", X_perfect_continuous, labels_random),
]:
    cv = cv_score(X, labels)
    eta2_trace = eta_squared_score(X, labels, method="trace")
    eta2_pf = eta_squared_score(X, labels, method="per_feature")
    print(f"\n{name}:")
    print(f"  CV (mean):              {cv.mean():.4f}  (lower = better)")
    print(f"  η² (trace):             {eta2_trace:.4f}  (higher = better)")
    print(f"  η² (per_feature):       {eta2_pf:.4f}  (higher = better)")

print(f"\n{'=' * 60}")
print("BINARY METRICS")
print("=" * 60)

for name, X, labels in [
    ("Perfect clusters", X_perfect_binary, labels_perfect),
    ("Moderate clusters", X_moderate_binary, labels_perfect),
    ("Random labels", X_random_binary, labels_random),
]:
    ent = entropy_score(X, labels)
    cv_val = cramers_v_score(X, labels)
    print(f"\n{name}:")
    print(f"  Entropy (mean):         {ent.mean():.4f}  (lower = better)")
    print(f"  Cramér's V (mean):      {cv_val:.4f}  (higher = better)")

CONTINUOUS METRICS


/Users/josh/miniforge3/envs/test/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: CV is unreliable when features contain non-positive values or have near-zero means. Consider using eta_squared_score instead.